# 00. Setup and tests

Clone the private repository into the Colab runtime, mount Google Drive, install `kidspipe`, set the environment, and run the self-test and the fast test suite.
Run this notebook once to check that a runtime works; the other notebooks repeat the same setup cell (a session starts from scratch every time).

| notebook | what it does |
|---|---|
| **00_setup_and_tests** | this one |
| 01_kv450_inference | KV450 xi+/- likelihood, MAP, MCMC, nested sampling: quick demos and the full chain |
| 02_cosebis_and_legacy | COSEBIs validation and the KiDS-Legacy comparison |
| 03_dr5_measurement | the full DR5 measurement chain from the public catalogue (resumable over many sessions) |

**Acknowledgement.** The DR5 / KiDS-Legacy data are KiDS products. Acknowledge KiDS and cite Wright et al. 2024 (A&A 686, A170) and the KiDS-Legacy papers
(<https://kids.strw.leidenuniv.nl/DR5/legacy_wl.php>); the repository's `README.md` has the full text. The repository is private; do not share the data or results
outside the terms of the KiDS release.

### Which runtime?

| | free Colab | Colab Pro / high-RAM |
|---|---|---|
| vCPU | 2 | about 8 |
| RAM | 12.7 GB | about 51 GB |
| session length | about 12 h (idle and background limits apply) | up to about 24 h |
| local disk | about 100 GB (wiped at the end of the session) | the same |
| Google Drive | persistent, shared 15 GB quota (free) | persistent |

The notebooks use `os.cpu_count()` for threads and worker processes. All times in these notebooks are **estimates** scaled from timings recorded on a
22-core development workstation (WSL2): linear scaling in cores is assumed and a Colab vCPU is assumed to be as fast as one of its cores, which is probably
optimistic. Treat them as lower bounds on a shared, noisy Colab host.

### One-time preparation: access to the private repository

The repository is private, so the notebook needs a GitHub **fine-grained personal access token**. The token is stored in **Colab Secrets**
and is never written into a cell, a file or the output.

1. GitHub, Settings, Developer settings, Personal access tokens, **Fine-grained tokens**, Generate new token. Resource owner: the account that owns the
   repository. Repository access: **Only select repositories**, choose this repository. Permissions: **Contents: Read-only** (Metadata is added
   automatically). Pick a short expiry. Copy the token once.
2. In Colab, left sidebar, the key icon (**Secrets**), Add new secret: name `GH_TOKEN`, value = the token, switch **Notebook access** on for this notebook
   (the switch is per notebook).
3. Edit `GITHUB_REPO` in the setup cell below (`owner/name`).

No GitHub access? Put a zip of the repository at `MyDrive/kidspipe/kidspipe_repo.zip`; the setup cell unpacks it instead of cloning.

## Session setup

A Colab runtime is fresh in every session: **run the next cell first, every time**. It mounts Drive, clones or updates the repository, installs the package if needed and sets the environment variables (`KIDSPIPE_RUNS`, `KIDSPIPE_DR5_WORK`, `KIDSPIPE_DR5_CATALOGUE`, `KIDSPIPE_PLOTS`, thread counts, numba cache on Drive) described in `docs/COLAB.md`.

In [ ]:
#@title 1. Session setup (run this first, in EVERY session) { display-mode: "form" }
#@markdown Private repository to clone (`owner/name`); the token comes from the Colab Secret `GH_TOKEN`.
GITHUB_REPO = "OWNER/REPO"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Persistent outputs live on Drive; the 16 GB catalogue lives on the local disk (never on Drive).
DRIVE_ROOT = "/content/drive/MyDrive/kidspipe"  #@param {type:"string"}
CATALOGUE_DIR = "/content/dr5"  #@param {type:"string"}
REPO_DIR = "/content/kidspipe"

import base64, importlib, os, pathlib, shutil, subprocess, sys

def _git(args, token=None, cwd=None):
    """git, with the token passed as an HTTP header (not stored in .git/config); the token is masked in anything returned."""
    cmd, secret = ["git"], []
    if token:
        b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {b64}"]
        secret = [token, b64]
    r = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secret:
        out = out.replace(s, "***")
    return r.returncode, out.strip()

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: Drive is not mounted; DRIVE_ROOT must be a writable directory.")

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception as e:                                   # secret missing, or notebook access not granted
    print(f"No GH_TOKEN secret available ({type(e).__name__}); using an existing clone or the Drive zip if there is one.")

repo = pathlib.Path(REPO_DIR)
url = f"https://github.com/{GITHUB_REPO}.git"
if (repo / "pyproject.toml").exists():
    rc, out = _git(["fetch", "--depth", "1", "origin", BRANCH], token, cwd=repo)
    if rc == 0:
        _git(["reset", "--hard", "FETCH_HEAD"], cwd=repo)
        print("repository updated to the latest", BRANCH)
    else:
        print("could not update (using the existing clone):", out[-300:])
elif token or not pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip").exists():
    if GITHUB_REPO.startswith("OWNER/"):
        raise SystemExit("Edit GITHUB_REPO (owner/name) in this cell, and add the GH_TOKEN secret (see the markdown above).")
    rc, out = _git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], token)
    if rc != 0:
        raise SystemExit("git clone failed (wrong GITHUB_REPO, or the token lacks Contents: read access?):\n" + out[-500:])
    print("cloned", GITHUB_REPO)
else:
    tmp = pathlib.Path("/content/_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip")), str(tmp))
    top = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(top[0]) if len(top) == 1 and not (tmp / "pyproject.toml").exists() else str(tmp), REPO_DIR)
    print("unpacked the repository from the Drive zip")

sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
import setup_env; importlib.reload(setup_env)
INFO = setup_env.setup(REPO_DIR, DRIVE_ROOT, CATALOGUE_DIR)      # env vars, pip install -e '.[all]' if needed, versions
os.chdir(REPO_DIR)
WORKERS = os.cpu_count() or 1

### What is where

* `/content/kidspipe`: the repository (local disk, recreated every session).
* `KIDSPIPE_RUNS = MyDrive/kidspipe/runs`: everything the pipeline writes (persistent). The reference results that ship in the repository (`/content/kidspipe/runs`) are read as
  fallback inputs (`kidspipe.paths.find_result`) but never overwritten.
* `KIDSPIPE_DR5_WORK = MyDrive/kidspipe/dr5_work`: the six per-bin npz of the DR5 extract (1.6 GB).
* `KIDSPIPE_DR5_CATALOGUE = /content/dr5/KiDS_Legacy_NS_unblind_final.fits`: the 16.4 GB catalogue, on local disk only.
* `NUMBA_CACHE_DIR = MyDrive/kidspipe/numba_cache`: compiled numba functions. The setup cell pins the source mtimes to the commit time so the cache stays valid across
  fresh clones of the same commit (first call of a fresh cache: about 10-30 s of JIT compilation; with a valid cache about 2 s).

In [ ]:
import os
for k in ("KIDSPIPE_ROOT", "KIDSPIPE_RUNS", "KIDSPIPE_DR5_WORK", "KIDSPIPE_DR5_CATALOGUE", "KIDSPIPE_PLOTS", "NUMBA_CACHE_DIR", "OMP_NUM_THREADS", "NUMBA_NUM_THREADS"):
    print(f"{k:24s} {os.environ.get(k)}")
from kidspipe import paths
print("\npaths.runs_dir()        ", paths.runs_dir())
print("paths.dr5_catalogue()   ", paths.dr5_catalogue())

### Self-test

`kidspipe check` evaluates chi2 at the spec test cosmology against the stored anchor (tolerance 0.5) and times the model stages. About 10-40 s including numba compilation.

In [ ]:
!python -m kidspipe check

### Fast test suite

`pytest -m 'not slow'`. The suite checks the theory chain, the likelihood, the samplers, COSEBIs, the KiDS-Legacy pipeline and the DR5 measurement code (the DR5 tests that need
data skip when it is absent). **Runtime estimate: roughly 10-20 min on 8 vCPU, 30-60 min on the free 2-vCPU runtime** (the full suite took 10-12 min on the 22-core development
workstation, and it uses worker processes in several tests). Wall-clock assertions carry the `timing` marker and are skipped on machines with fewer than 8 cores unless
`KIDSPIPE_TIMING_TESTS=1`; no tolerance is relaxed for slow machines. Use `-x` to stop at the first failure.

In [ ]:
!python -m pytest -q -m 'not slow' -p no:cacheprovider --durations=10

Optional: the tests marked `slow` (validation against high-accuracy references; long on 2 vCPU, hours possible).

In [ ]:
RUN_SLOW = False   #@param {type:"boolean"}
if RUN_SLOW:
    !python -m pytest -q -m 'slow' -p no:cacheprovider --durations=10

### Record the environment on Drive (provenance of every later result)

In [ ]:
import pathlib
runs = pathlib.Path(os.environ["KIDSPIPE_RUNS"])
!pip freeze > {runs}/pip_freeze.txt
!git -C {REPO_DIR} log -1 --format='%H %cd %s' > {runs}/commit.txt
print((runs / "commit.txt").read_text()); print("saved", runs / "pip_freeze.txt", "and", runs / "session_info.json")

### Next

* `01_kv450_inference.ipynb`: quick demos first (minutes), then the full chain.
* `02_cosebis_and_legacy.ipynb`: COSEBIs validation and the KiDS-Legacy comparison (the shipped KV450 chains are used when you have not re-run the KV450 chain).
* `03_dr5_measurement.ipynb`: the DR5 measurement from the public catalogue. `!python scripts/dr5_chain.py status` shows at any time what is done and what is left.

In [ ]:
!python scripts/dr5_chain.py status --threads {WORKERS}